# Data Preprocessing

## 1. Import raster and vector libraries

In [ ]:
import rioxarray as rxr
import geopandas as gpd

## 2. Open PRODES 2024 raster and inspect CRS

In [ ]:
prodes = rxr.open_rasterio('../inputs/PRODES/prodes_brasil_2024.tif', masked=True)
print(prodes.rio.crs)

## 3. Load Pará boundary and inspect CRS

In [ ]:
para = gpd.read_file('../inputs/para_boundary/para.gpkg')
print(para.crs)

## 4. Clip and reproject PRODES raster to Pará (EPSG:31982)

In [ ]:
!gdalwarp \
-t_srs EPSG:31982 \
-cutline ../inputs/para_boundary/para.gpkg \
-crop_to_cutline \
-dstnodata -9999 \
-ot Int16 \
../inputs/PRODES/prodes_brasil_2024.tif \
../inputs/label/prodes_para.tif

## 5. Generate binary deforestation label raster

In [ ]:
!gdal_calc.py \
-A ../inputs/label/prodes_para.tif \
--outfile=../inputs/label/para_two_class.tif \
--calc="(A>=0)*(A<=64)" \
--type=Int16 \
--NoDataValue=-9999 \
--quiet

## 6. Build Landsat QA cloud-band virtual raster

In [ ]:
!ls ../inputs/landsat_rgb_qa/*.tif | sort | \
gdalbuildvrt \
-b 7 \
-input_file_list /vsistdin/ \
qa_cloud.vrt

## 7. Align and clip QA cloud raster to binary label grid

In [ ]:
%%bash
gdalwarp \
  -t_srs EPSG:31982 \
  -r near \
  -cutline ../inputs/para_boundary/para.gpkg \
  -crop_to_cutline \
  -te $(gdalinfo -json ../inputs/label/para_two_class.tif | python -c "import json,sys; j=json.load(sys.stdin); cc=j['cornerCoordinates']; xmin,ymax=cc['upperLeft']; xmax,ymin=cc['lowerRight']; print(xmin,ymin,xmax,ymax)") \
  -ts $(gdalinfo -json ../inputs/label/para_two_class.tif | python -c "import json,sys; j=json.load(sys.stdin); print(j['size'][0], j['size'][1])") \
  -ot Int16 \
  -dstnodata -9999 \
  qa_cloud.vrt \
  qa_crop.vrt

## 8. Generate three-class label raster (non-deforestation, deforestation, cloud)

In [ ]:
!gdal_calc.py \
-A ../inputs/label/para_two_class.tif \
-B qa_crop.vrt \
--outfile=../inputs/label/para_three_class.tif \
--calc="where(B==1, 2, A)" \
--type=Int16 \
--NoDataValue=-9999 \
--quiet

## 9. Build six-band Landsat imagery virtual raster

In [ ]:
!ls ../inputs/landsat_rgb_qa/*.tif | sort | \
gdalbuildvrt \
-b 1 -b 2 -b 3 -b 4 -b 5 -b 6\
-input_file_list /vsistdin/ \
rgb.vrt

## 10. Align and export six-band Landsat imagery to label grid

In [ ]:
%%bash

gdalwarp \
  -of GTiff \
  -t_srs EPSG:31982 \
  -r bilinear \
  -cutline ../inputs/para_boundary/para.gpkg \
  -crop_to_cutline \
  -te $(gdalinfo -json ../inputs/label/para_three_class.tif | python -c "import json,sys; j=json.load(sys.stdin); cc=j['cornerCoordinates']; xmin,ymax=cc['upperLeft']; xmax,ymin=cc['lowerRight']; print(xmin,ymin,xmax,ymax)") \
  -ts $(gdalinfo -json ../inputs/label/para_three_class.tif | python -c "import json,sys; j=json.load(sys.stdin); print(j['size'][0], j['size'][1])") \
  -ot Int16 \
  -dstnodata -9999 \
  rgb.vrt \
  ../inputs/map/map.tif